# ComfyUI Drama Production Runtime

This notebook installs a pinned, restartable render environment for the book-to-video project. It deliberately exposes **no public ComfyUI UI or tunnel**; orchestration calls the REST API on `127.0.0.1` only.

Coverage map:

| Workstream | Trigger → output | Done when |
|---|---|---|
| Bootstrap | pinned project URL/SHA → source checkout | HEAD equals requested SHA |
| Persistence | optional Drive mount → workspace/output paths | restart preserves selected data |
| Install | reviewed manifest → pinned repos, nodes, models | checksums and required paths pass |
| Runtime | local process → `127.0.0.1:8188` | health and node-class preflight pass |
| Render | project `spec.yaml` → drama-video stages | status can resume completed artifacts |
| Delivery | local output tree → Drive | media/spec/status artifacts copied |

**Expected input later:** a project folder containing the generated `spec.yaml`, images, and Edge-TTS audio/cues.

## 1. Runtime configuration

Select a GPU runtime before continuing. Use exact 40-character commit SHAs. Branch names such as `main` are intentionally rejected.

In [ ]:
#@title Immutable bootstrap inputs
PROJECT_REPO_URL = "https://github.com/Yuki-TR/mobydick.git"  #@param {type:"string"}
PROJECT_COMMIT = "559db35b43ce57d85dd39bc0442b47002feae822"  #@param {type:"string"}
USE_GOOGLE_DRIVE = True  #@param {type:"boolean"}
DRIVE_FOLDER = "ComfyDrama"  #@param {type:"string"}
MANIFEST_RELATIVE_PATH = "scripts/colab/manifest.production.json"  #@param {type:"string"}

import re
if not PROJECT_REPO_URL.startswith("https://"):
    raise ValueError("PROJECT_REPO_URL must be an HTTPS git URL")
if not re.fullmatch(r"[0-9a-fA-F]{40}", PROJECT_COMMIT):
    raise ValueError("PROJECT_COMMIT must be an exact 40-character SHA")


In [ ]:
# Pinned project checkout (safe to re-run)
import pathlib, subprocess
SOURCE_ROOT = pathlib.Path("/content/comfy-drama-src")
if not (SOURCE_ROOT / ".git").exists():
    if SOURCE_ROOT.exists() and any(SOURCE_ROOT.iterdir()):
        raise RuntimeError(f"Refusing to replace non-git directory: {SOURCE_ROOT}")
    SOURCE_ROOT.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "init"], cwd=SOURCE_ROOT, check=True)
    subprocess.run(["git", "remote", "add", "origin", PROJECT_REPO_URL], cwd=SOURCE_ROOT, check=True)
else:
    origin = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=SOURCE_ROOT, text=True).strip()
    if origin.rstrip("/") != PROJECT_REPO_URL.rstrip("/"):
        raise RuntimeError(f"Existing checkout has a different origin: {origin}")
subprocess.run(["git", "fetch", "--depth", "1", "origin", PROJECT_COMMIT], cwd=SOURCE_ROOT, check=True)
# Do not force checkout: persistent user edits must cause a visible failure, not be overwritten.
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=SOURCE_ROOT, check=True)
actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=SOURCE_ROOT, text=True).strip()
assert actual.lower() == PROJECT_COMMIT.lower(), (actual, PROJECT_COMMIT)
print("Project checkout ready:", actual)

In [ ]:
# Optional Drive persistence and deterministic paths
import os, pathlib
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = pathlib.Path("/content/drive/MyDrive") / DRIVE_FOLDER
    COLAB_WORKSPACE = DRIVE_ROOT / "runtime"
    DELIVERY_ROOT = DRIVE_ROOT / "deliveries"
else:
    DRIVE_ROOT = None
    COLAB_WORKSPACE = pathlib.Path("/content/comfy-drama-runtime")
    DELIVERY_ROOT = pathlib.Path("/content/comfy-drama-deliveries")
COLAB_WORKSPACE.mkdir(parents=True, exist_ok=True)
DELIVERY_ROOT.mkdir(parents=True, exist_ok=True)
os.environ["COLAB_WORKSPACE"] = str(COLAB_WORKSPACE)
os.environ["COMFY_URL"] = "http://127.0.0.1:8188"
print({"workspace": str(COLAB_WORKSPACE), "deliveries": str(DELIVERY_ROOT)})

## 2. Reviewed install manifest

The production manifest must contain official asset URLs, exact custom-node commits, destinations, and available SHA-256 checksums. Installation is blocked while `manifest_complete` is false. Tokens belong in Colab Secrets/environment variables, never in the manifest.

In [ ]:
# Select/copy the manifest into persistent configuration
import shutil
manifest_source = SOURCE_ROOT / MANIFEST_RELATIVE_PATH
if not manifest_source.is_file():
    template = SOURCE_ROOT / "scripts/colab/manifest.template.json"
    raise FileNotFoundError(
        f"Reviewed production manifest not found: {manifest_source}. "
        f"Start from {template}; install will remain blocked until it is complete."
    )
MANIFEST_PATH = COLAB_WORKSPACE / "config/manifest.json"
MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(manifest_source, MANIFEST_PATH)
print("Using manifest:", MANIFEST_PATH)

In [ ]:
# Host preflight: GPU, disk, git, ffmpeg, manifest warnings
RUNTIME = SOURCE_ROOT / "scripts/colab/colab_runtime.py"
subprocess.run(["python3", str(RUNTIME), "--manifest", str(MANIFEST_PATH), "validate"], check=True)
subprocess.run(["python3", str(RUNTIME), "--manifest", str(MANIFEST_PATH), "host-preflight"], check=True)

In [ ]:
# Install pinned repos/nodes and verified model assets; idempotent on restart
subprocess.run(["python3", str(RUNTIME), "--manifest", str(MANIFEST_PATH), "install"], check=True)

## 3. Start private ComfyUI and verify the workflow surface

The helper rejects `0.0.0.0` and non-local URLs. The notebook intentionally has no UI/tunnel cell.

In [ ]:
subprocess.run(["python3", str(RUNTIME), "--manifest", str(MANIFEST_PATH), "launch", "--host", "127.0.0.1", "--port", "8188"], check=True)
subprocess.run(["python3", str(RUNTIME), "--manifest", str(MANIFEST_PATH), "wait", "--url", os.environ["COMFY_URL"], "--timeout", "600"], check=True)
subprocess.run(["python3", str(RUNTIME), "--manifest", str(MANIFEST_PATH), "runtime-preflight", "--url", os.environ["COMFY_URL"]], check=True)

In [ ]:
# Diagnostics: last server log lines (does not expose ComfyUI publicly)
log_path = COLAB_WORKSPACE / ".colab_state/comfyui.log"
if log_path.exists():
    print("\n".join(log_path.read_text(errors="replace").splitlines()[-80:]))

## 4. Project dry run

By default this extracts the ready Moby-Dick pilot bundled in the repository. `plan` validates its YAML and timing without rendering. Set `PROJECT_DIR` only when you want to render a different prepared project.

In [ ]:
#@title Project input
PROJECT_DIR = ""  #@param {type:"string"}
if PROJECT_DIR.strip():
    project_dir = pathlib.Path(PROJECT_DIR).expanduser().resolve()
else:
    import shutil
    archive = SOURCE_ROOT / "deliverables/moby_dick_pilot_colab.zip"
    extracted = pathlib.Path("/content/moby_dick_pilot")
    if not archive.is_file():
        raise FileNotFoundError(f"Bundled pilot archive not found: {archive}")
    if not extracted.exists():
        extracted.mkdir(parents=True)
        shutil.unpack_archive(str(archive), str(extracted), "zip")
    project_dir = extracted / "build"
spec_path = project_dir / "spec.yaml"
if not spec_path.is_file():
    raise FileNotFoundError(f"Project spec not found: {spec_path}")
import json
manifest_data = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
creative_entry = next((item for item in manifest_data["repositories"] if item["name"].lower() == "creative-skills"), None)
if creative_entry is None:
    raise RuntimeError("Manifest has no creative-skills repository entry")
creative_root = COLAB_WORKSPACE / creative_entry["destination"]
drama_script = creative_root / "drama-video/scripts/drama_video.py"
if not drama_script.is_file():
    raise FileNotFoundError(f"Drama orchestrator not installed: {drama_script}")
print({"project": str(project_dir), "spec": str(spec_path)})

In [ ]:
# Side-effect-free plan/dry run
subprocess.run(["python3", str(drama_script), "plan", str(spec_path)], cwd=project_dir, check=True)

## 5. Render or resume

Use `status` after reconnecting. Prefer rendering one shot first; move to `all` only after the pilot output is accepted. Existing artifacts are preserved by the upstream restart-safe orchestrator.

In [ ]:
#@title Render stage
STAGE = "status"  #@param ["status", "plan", "anchors", "shot", "shots", "assemble", "all"]
SHOT_NUMBER = 1  #@param {type:"integer"}
allowed = {"status", "plan", "anchors", "shot", "shots", "assemble", "all"}
if STAGE not in allowed:
    raise ValueError(f"Unsupported stage: {STAGE}")
command = ["python3", str(drama_script), STAGE]
if STAGE == "shot":
    command.append(str(SHOT_NUMBER))
command.append(str(spec_path))
subprocess.run(command, cwd=project_dir, env={**os.environ, "COMFY_URL": "http://127.0.0.1:8188"}, check=True)

In [ ]:
# Explicit resume snapshot
subprocess.run(["python3", str(RUNTIME), "--manifest", str(MANIFEST_PATH), "status", "--project-dir", str(project_dir)], check=True)

## 6. Deliver outputs

Copies media and sidecars without deleting local or Drive files. With Drive disabled, the destination is `/content/comfy-drama-deliveries` and must be downloaded before the runtime ends.

In [ ]:
delivery = DELIVERY_ROOT / project_dir.name
subprocess.run(["python3", str(RUNTIME), "--manifest", str(MANIFEST_PATH), "copy-outputs", "--source", str(project_dir), "--destination", str(delivery)], check=True)
print("Delivery ready:", delivery)

## Recovery notes

- Reconnect: rerun configuration, checkout, Drive, manifest, launch, then `status`.
- Download interrupted: rerun install; `.part` files are discarded and verified files are skipped.
- Node missing: inspect the server log, correct the pinned manifest, restart the runtime, and rerun runtime preflight.
- Out of memory: render one short shot at the pilot resolution; do not change model files mid-project without producing a new manifest.
- No public browser UI is part of this deployment. All render traffic stays inside the Colab VM.